# BeRL — Comprehensive, eval-grounded re-assessment of all Completed runs

**Goal:** find a winning *training recipe* by ranking every Completed phase-stability run on
**authoritative eval accuracy** — the full `subsample300` `val/test_score/<bench>` numbers —
instead of training reward / rollout response-length, which are **misleading**.

### Why not reward / resp_len (the PS129 lesson)
PS129 (Gemma-2, power-k7, actor-RM) had the single biggest HM jump (0.085→0.43) but:
its training rollout `resp_len` **collapsed to ~1.5 tokens**, `kl` blew up to the ~10 cap, and
reward inflated to 15.7 (actor-as-RM self-inflation). Yet its *eval* CoTs stayed coherent and
gsm8k/mmlu **rose** — so the eval gain was real but the run is training-unstable / reward-hacks
the behavior objective. **Conclusion: rank on eval accuracy, gate on eval-CoT quality, and treat
reward/KL/resp_len as stability *annotations*, not the score.**

### Metric definitions
* **ToM HM** = harmonic mean over the 24 ToM benchmarks (excludes gsm8k, mmlu), avg-then-HM over
  the last-N eval iters (identical convention to `scripts/score_run.py`).
* **dHM** = `HM(last3) - HM(step0)` — vs each run's OWN step-0 baseline, ranked **within model
  family** (Qwen2.5 base HM≈0.42, Qwen3≈0.34/0.16, Gemma-2≈0.09 are not comparable in absolute).
* **stability** = std of per-iter ToM HM over the last 5 evals (prefer a sustained plateau).
* **regression** = dgsm8k, dmmlu (reported separately, never folded into HM).
* **eval-quality gate** = format-parseable & correct fraction from `[val sample|step|score]` blocks,
  plus eval-CoT char length (catches 'empty CoT wins binary MC by luck', which test_score alone can't).
* **health annotation** = final KL (flag near the ~10 cap), min rollout resp_len (collapse).


## 0. Setup
Run from the repo root in the `tom` conda env. The heavy lifting lives in
`scripts/reassess_runs.py` (rg-accelerated extraction, cached to `analysis/cache/`).


In [1]:
import sys, os, importlib
# locate repo root (dir containing scripts/reassess_runs.py) so the notebook runs from anywhere
root = os.getcwd()
while root != '/' and not os.path.exists(os.path.join(root,'scripts','reassess_runs.py')):
    root = os.path.dirname(root)
os.chdir(root); sys.path.insert(0, os.path.join(root,'scripts'))
print('repo root:', root)
import reassess_runs as R
importlib.reload(R)
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

repo root: /mnt/home/judekhouja/repo/BeRL


## 1. Extract + score every Completed run
Resolves each Completed tracker row to its disk log (picking the retry attempt with the most eval
iters), extracts eval trajectories + health + val-sample quality, and computes the metrics.
First run parses ~72 GB (a few minutes); results are cached so reruns are instant.


In [2]:
rows = R.assess_all(status='completed')
df = pd.DataFrame(rows)
df = df[df['n_iters'].fillna(0) >= 2].copy()
print(len(df), 'scored runs')
df.to_csv('analysis/reassess.csv', index=False)


[1/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp0_ec0.0_lp-dcfg_smoke_mix-Q
[2/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp0_ec0.001_lp-dcfg_smoke_mix
[3/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp5_ec0.0_lp-dcfg_smoke_mix-Q
[4/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp5_ec0.001_lp-dcfg_smoke_mix
[5/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp0_ec0.0_lp-dcfg_smoke_mix-Q
[6/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp0_ec0.001_lp-dcfg_smoke_mix
[7/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp5_ec0.0_lp-dcfg_smoke_mix-Q
[8/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp5_ec0.001_lp-dcfg_smoke_mix
[9/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_ec0.0_lp-dcfg_smoke_mix-Q
[10/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_ec0.001_lp-dcfg_smoke_mix
[11/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_ec0.0_lp-dcfg_smoke_mix-Q
[12/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_ec0.001_lp-dcfg_smoke_mix
[13/152] Phase-stability-Pm1swp_rmf_kl0.05_lr1e-6

[83/152] Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp0_ec0.001_pk5-dcfg_smoke_mi
[84/152] Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_ec0.0_pk5-dcfg_smoke_mix-
[85/152] Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_ec0.001_pk5-dcfg_smoke_mi
[86/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr1e-6_fp0_ec0.0_lp-dcfg_smoke_
[87/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr1e-6_fp0_ec0.001_lp-dcfg_smok
[88/152] Phase-stability-Pm1w2gemma2_rma_kl0.05_lr1e-6_fp0_ec0.0_lp-dcfg_smoke_
[89/152] Phase-stability-Pm1w2gemma2_rma_kl0.05_lr1e-6_fp5_ec0.001_lp-dcfg_smok
[90/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp0_ec0.0_pk5_llm6-dcfg_
[91/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp5_ec0.0_pk5_llm6-dcfg_
[92/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp5_ec0.001_pk5_llm6-dcf
[93/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp0_ec0.0_pk5_llm4-dcfg_
[94/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp0_ec0.001_pk5_llm4-dcf
[95/152] Phase-stability-Pm1w2gemma2_rmf

152 scored runs


## 2. Health gate (clean vs reward-hacking-unstable)
A run is **clean** if it neither blew up KL (`kl_final < 1.0`) nor collapsed rollouts
(`resp_len_min >= 30`). This is an annotation used to separate *adoptable* recipes from unstable
outliers like PS129 — it does **not** remove them from the eval-accuracy ranking.


In [3]:
for c in ['d_hm','d_avg','kl_final','resp_len_min','d_gsm8k','d_mmlu','hm_step0','hm_last3','hm_late_std']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['clean'] = (df['kl_final'] < 1.0) & (df['resp_len_min'] >= 30)
print('frozen RM  KL-blowups:', ((df.rm_mode=='frozen') & (df.kl_final>1)).sum(), '/', (df.rm_mode=='frozen').sum())
print('actor  RM  KL-blowups:', ((df.rm_mode=='actor')  & (df.kl_final>1)).sum(), '/', (df.rm_mode=='actor').sum())


frozen RM  KL-blowups: 0 / 80
actor  RM  KL-blowups: 12 / 72


## 3. Per-family leaderboard (ranked by dHM)
Absolute HM is dominated by base model, so we rank **within** family.


In [4]:
cols = ['run','reward','power_k','ll_min','rm_mode','kl','lr','fp','ec',
        'd_hm','d_avg','hm_step0','hm_last3','hm_late_std','ecorr_late','d_gsm8k','d_mmlu',
        'kl_final','resp_len_min','clean','n_iters']
for fam in ['Qwen2.5','Qwen3','Gemma-2']:
    print(f'\n===== {fam} — top 10 by dHM =====')
    sub = df[df.model==fam].sort_values('d_hm', ascending=False)[cols].head(10)
    display(sub)



===== Qwen2.5 — top 10 by dHM =====


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,ecorr_late,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
150,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_e...,power,7.0,6.0,frozen,0.05,5e-7,0.0,0.001,0.0547,0.0236,0.4149,0.4696,0.0016,0.4643,0.0043,0.1413,0.084,50.2,True,14
82,Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp0_e...,power,5.0,6.0,actor,0.05,1e-6,0.0,0.001,0.0541,0.0235,0.4158,0.4699,0.0060,0.5046,-0.0433,0.1480,0.063,43.4,True,19
80,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp5_e...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.000,0.0529,0.0202,0.4158,0.4687,0.0018,0.5417,-0.0090,0.1613,0.093,39.6,True,20
84,Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_e...,power,5.0,6.0,actor,0.05,1e-6,5.0,0.001,0.0526,0.0214,0.4166,0.4693,0.0033,0.4917,-0.0323,0.1617,0.089,40.6,True,20
83,Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_e...,power,5.0,6.0,actor,0.05,1e-6,5.0,0.000,0.0520,0.0207,0.4150,0.4670,0.0032,0.5208,0.0057,0.1633,0.098,46.3,True,20
70,Phase-stability-Pm1swp_rmf_kl0.05_lr1e-6_fp5_e...,power,5.0,6.0,frozen,0.05,1e-6,5.0,0.000,0.0501,0.0222,0.4178,0.4679,0.0049,0.5125,-0.0990,0.1367,0.087,44.6,True,20
81,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp5_e...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.001,0.0491,0.0214,0.4159,0.4650,0.0055,0.5458,-0.0090,0.1453,0.084,43.9,True,20
79,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp0_e...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0483,0.0197,0.4154,0.4637,0.0025,0.5292,-0.0323,0.1443,0.100,42.0,True,20
67,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_e...,power,5.0,6.0,frozen,0.05,5e-7,5.0,0.001,0.0478,0.0217,0.4208,0.4686,0.0012,0.5602,-0.0083,0.1670,0.092,48.8,True,19
77,Phase-stability-Pm1swp_rma_kl0.01_lr1e-6_fp5_e...,power,5.0,6.0,actor,0.01,1e-6,5.0,0.001,0.0478,0.0161,0.4156,0.4633,0.0062,0.4792,-0.0900,0.1373,0.160,40.5,True,20



===== Qwen3 — top 10 by dHM =====


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,ecorr_late,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
115,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr1e-6_f...,log_prob,2.0,8.0,frozen,0.05,1e-6,0.0,0.001,0.0632,0.0397,0.3415,0.4046,0.0089,0.4107,0.0173,0.0607,0.011,608.9,True,14
114,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr1e-6_f...,log_prob,2.0,8.0,frozen,0.05,1e-6,0.0,0.000,0.0585,0.0377,0.3403,0.3988,0.0053,0.4762,0.0043,0.0490,0.014,622.6,True,14
142,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,4.0,actor,0.05,5e-7,5.0,0.000,0.0339,0.0256,0.3342,0.3681,0.0182,0.4042,0.0180,0.0370,0.002,566.6,True,39
139,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.001,0.0185,0.0076,0.3422,0.3607,0.0076,0.4292,0.0177,0.0117,0.002,592.9,True,39
124,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,5.0,6.0,frozen,0.05,5e-7,5.0,0.001,0.0103,0.0044,0.3423,0.3526,0.0115,0.4833,0.0120,0.0040,0.003,567.2,True,39
131,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,7.0,6.0,frozen,0.05,5e-7,5.0,0.000,0.0093,0.0050,0.3419,0.3512,0.0141,0.4667,0.0007,-0.0077,0.004,586.0,True,39
137,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0069,-0.0021,0.1605,0.1674,0.0038,0.2708,-0.0157,0.0297,0.003,439.1,True,20
129,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,7.0,6.0,frozen,0.05,5e-7,0.0,0.000,0.0068,0.0036,0.1605,0.1673,0.0063,0.2708,-0.0003,0.0447,0.003,438.2,True,39
145,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,7.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0048,0.0044,0.1607,0.1655,0.0140,0.3000,0.0207,0.0257,0.003,430.3,True,39
138,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.000,-0.0011,-0.0029,0.1605,0.1594,0.0060,0.2417,-0.0040,0.0117,0.003,432.0,True,39



===== Gemma-2 — top 10 by dHM =====


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,ecorr_late,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
109,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,7.0,6.0,actor,0.05,5e-7,0.0,0.000,0.3445,0.1919,0.0853,0.4298,0.0068,0.4750,0.1530,0.1163,9.852,1.5,False,20
92,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,0.05,5e-7,0.0,0.000,0.1326,0.1041,0.0930,0.2256,0.0253,0.3917,0.0897,0.0623,0.095,64.6,True,39
108,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,4.0,actor,0.05,5e-7,5.0,0.001,0.1159,0.0807,0.0929,0.2088,0.0203,0.4083,0.0963,0.0467,0.029,61.9,True,39
105,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.001,0.1014,0.0998,0.0930,0.1944,0.0439,0.4542,0.2507,0.0907,0.580,10.3,False,38
103,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.000,0.0886,0.0643,0.0854,0.1740,0.0510,0.3250,0.1947,0.0750,1.845,21.4,False,20
104,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0818,0.0885,0.0930,0.1748,0.0273,0.3375,0.2067,0.0497,3.392,19.4,False,39
113,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,7.0,4.0,actor,0.05,5e-7,5.0,0.000,0.0795,0.0756,0.0930,0.1725,0.0390,0.3875,0.0553,0.0577,0.042,65.0,True,39
95,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,0.05,5e-7,5.0,0.001,0.0751,0.0968,0.0930,0.1681,0.0246,0.4167,0.0563,0.0257,0.077,84.0,True,39
110,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,7.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0619,0.0485,0.0930,0.1549,0.0222,0.2917,0.2163,0.0787,0.553,27.6,False,39
100,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,7.0,4.0,frozen,0.05,5e-7,0.0,0.001,0.0552,0.0859,0.0930,0.1482,0.0426,0.4000,0.0830,0.0367,0.059,69.1,True,20


## 4. Clean winner per family (health-gated)
The best *adoptable* recipe per family = highest dHM among `clean` runs.


In [5]:
winners = (df[df.clean].sort_values('d_hm', ascending=False)
             .groupby('model', as_index=False).first())
display(winners[cols])


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,ecorr_late,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
0,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,0.05,5e-7,0.0,0.000,0.1326,0.1041,0.0930,0.2256,0.0253,0.3917,0.0897,0.0623,0.095,64.6,True,39
1,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_e...,power,7.0,6.0,frozen,0.05,5e-7,0.0,0.001,0.0547,0.0236,0.4149,0.4696,0.0016,0.4643,0.0043,0.1413,0.084,50.2,True,14
2,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr1e-6_f...,log_prob,2.0,8.0,frozen,0.05,1e-6,0.0,0.001,0.0632,0.0397,0.3415,0.4046,0.0089,0.4107,0.0173,0.0607,0.011,608.9,True,14


## 5. Winning archetype (reward x RM), clean only
Which knob combinations robustly help vs hurt.


In [6]:
arch = (df[df.clean].groupby(['model','reward','rm_mode'])['d_hm']
          .agg(['count','mean','max']).reset_index()
          .sort_values('mean', ascending=False))
display(arch[arch['count']>=2])


,model,reward,rm_mode,count,mean,max
2,Gemma-2,power,actor,4,0.041400,0.1159
6,Qwen2.5,power,actor,25,0.031872,0.0541
9,Qwen3,log_prob,frozen,4,0.007300,0.0632
3,Gemma-2,power,frozen,11,0.003509,0.1326
7,Qwen2.5,power,frozen,30,0.002867,0.0547
10,Qwen3,power,actor,12,-0.012167,0.0339
0,Gemma-2,log_prob,actor,2,-0.014500,-0.0026
11,Qwen3,power,frozen,14,-0.021043,0.0103
4,Qwen2.5,log_prob,actor,11,-0.036473,0.0268
5,Qwen2.5,log_prob,frozen,16,-0.082825,0.0163


## 6. Eval-CoT quality gate (anti-MC-luck)
Confirm the winners emit real, non-empty eval CoTs (not short outputs winning binary MC by chance).
Extracts the actual generation after the assistant turn marker from `[val sample]` blocks.


In [7]:
import re
HDR = re.compile(r'\[val sample \| step=(\d+) \| source=(\w+) \| score=(-?[0-9.]+)\]')
def eval_gen_lengths(path):
    lines=[re.sub(r'^\(main_task pid=\d+\)\s?','',l) for l in open(path,encoding='utf-8',errors='ignore')]
    n=len(lines); i=0; per={}
    while i<n:
        m=HDR.search(lines[i])
        if m:
            step=int(m.group(1)); j=i+1; gen=None
            while j<n and not HDR.search(lines[j]):
                s=lines[j].strip()
                if s.endswith('<start_of_turn>model') or s.endswith('assistant'):
                    k=j+1
                    while k<n and lines[k].strip()=='': k+=1
                    gen=re.sub(r'^\(main_task pid=\d+\)\s?','',lines[k]).strip() if k<n else ''
                    break
                j+=1
            per.setdefault(step,[]).append(len(gen) if gen else 0)
            i=j
        else: i+=1
    return per
for _,w in winners.iterrows():
    per=eval_gen_lengths(w['log']); steps=sorted(per)
    if not steps: continue
    last=[x for s in steps[-3:] for x in per[s]]
    print(f"{w['model']:8} eval-CoT chars: step0={np.mean(per[steps[0]]):.0f} last3={np.mean(last):.0f} "
          f"empty(<5)={sum(1 for x in last if x<5)}/{len(last)}  | {w['reward']} k{w['power_k']} {w['rm_mode']}RM")


Gemma-2  eval-CoT chars: step0=144 last3=408 empty(<5)=0/78  | power k5.0 frozenRM


Qwen2.5  eval-CoT chars: step0=452 last3=445 empty(<5)=0/78  | power k7.0 frozenRM


Qwen3    eval-CoT chars: step0=39 last3=39 empty(<5)=0/78  | log_prob k2.0 frozenRM


## 8. Format vs reasoning decomposition (is the gain just formatting?)

Each eval score decomposes **exactly**: `score = format(±1) + answer(±2)` →
`+3`=parsed & correct (what `val/test_score` counts), `−1`=parsed but wrong, `−3`=format-fail.
So from the `[val sample|source|score]` blocks (ToM-only) we split the correct-rate change into:
* **format-pass rate** — fraction of eval responses that parse to a valid answer;
* **conditional accuracy** `P(correct | parsed)` — pooled (`d_cond_acc`) and per-benchmark then
  **arithmetic-meaned** (`d_cavg` = *arithmetic-mean accuracy conditional on correct format*);
* a multiplicative split of the correct-rate gain into `fmt_pct` (format-driven) vs `reason_pct`.

`d_cavg` / `d_cond_acc` are the **format-controlled ToM-reasoning** signal. **Caveat:** they come
from the small 24-sample/step val-debug blocks (≈240 samples/window, SE≈0.05), so *single-run*
values are noisy and small effects can flip sign vs the full-eval HM — trust the **archetype
aggregation** below, not individual rows. (A cleaner fix would be to log full-eval format-pass in
`ray_trainer._validate`.)


In [8]:
eqcols = ['run','reward','power_k','ll_min','rm_mode','fp','lr','kl',
          'd_hm','d_avg','d_cavg','d_cond_acc','d_fmt_pass','fmt_pct','reason_pct','d_gsm8k','clean']
for c in ['d_cavg','d_cond_acc','d_fmt_pass','cavg_early','cavg_late']:
    df[c] = pd.to_numeric(df[c], errors='coerce')


### 8a. Do the raw-HM winners survive format control?
Compare each family's raw-HM winner on `d_hm` vs the format-controlled `d_cavg`/`d_cond_acc`.


In [9]:
raw_win = (df[df.clean].sort_values('d_hm', ascending=False).groupby('model', as_index=False).first())
display(raw_win[['model','reward','power_k','ll_min','rm_mode','fp','d_hm','d_avg','d_cavg','d_cond_acc','d_fmt_pass','fmt_pct','reason_pct']])


,model,reward,power_k,ll_min,rm_mode,fp,d_hm,d_avg,d_cavg,d_cond_acc,d_fmt_pass,fmt_pct,reason_pct
0,Gemma-2,power,5.0,4.0,frozen,0.0,0.1326,0.1041,-0.0096,-0.0247,0.0917,145.3,-45.3
1,Qwen2.5,power,7.0,6.0,frozen,0.0,0.0547,0.0236,-0.0933,-0.0974,0.0238,-16.1,116.1
2,Qwen3,log_prob,2.0,8.0,frozen,0.0,0.0632,0.0397,-0.1077,-0.0966,0.0417,-40.1,140.1


### 8b. Re-rank on conditional accuracy (`d_cavg`, format-controlled), clean only


In [10]:
for fam in ['Qwen2.5','Qwen3','Gemma-2']:
    print(f'\n===== {fam} — top 5 by d_cavg (format-controlled) =====')
    sub = df[(df.model==fam) & df.clean].sort_values('d_cavg', ascending=False)[eqcols].head(5)
    display(sub)



===== Qwen2.5 — top 5 by d_cavg (format-controlled) =====


,run,reward,power_k,ll_min,rm_mode,fp,lr,kl,d_hm,d_avg,d_cavg,d_cond_acc,d_fmt_pass,fmt_pct,reason_pct,d_gsm8k,clean
39,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_e...,power,3.0,6.0,frozen,5.0,5e-7,0.05,0.0301,0.0060,0.0969,0.0928,0.0000,0.0,100.0,-0.1723,True
73,Phase-stability-Pm1swp_rma_kl0.01_lr5e-7_fp5_e...,power,5.0,6.0,actor,5.0,5e-7,0.01,0.0391,0.0114,0.0808,0.0793,0.0167,9.4,90.6,-0.0337,True
25,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp0_e...,log_prob,2.0,8.0,actor,0.0,5e-7,0.05,0.0268,-0.0044,0.0703,0.0678,0.0000,0.0,100.0,-0.0583,True
80,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp5_e...,power,5.0,6.0,actor,5.0,5e-7,0.05,0.0529,0.0202,0.0473,0.0459,0.0167,15.3,84.7,-0.0090,True
11,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_e...,log_prob,2.0,8.0,frozen,5.0,5e-7,0.05,0.0058,-0.0182,0.0472,0.0445,-0.0125,-16.2,116.2,-0.1297,True



===== Qwen3 — top 5 by d_cavg (format-controlled) =====


,run,reward,power_k,ll_min,rm_mode,fp,lr,kl,d_hm,d_avg,d_cavg,d_cond_acc,d_fmt_pass,fmt_pct,reason_pct,d_gsm8k,clean
143,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,4.0,actor,5.0,5e-7,0.05,-0.0082,-0.0043,0.1179,0.0141,0.0292,59.7,40.3,-0.0023,True
147,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,7.0,4.0,actor,5.0,5e-7,0.05,-0.0429,-0.0131,0.1019,0.0174,-0.0333,304.8,-204.8,-0.0503,True
120,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr1e-6_f...,log_prob,2.0,8.0,actor,5.0,1e-6,0.05,-0.0990,-0.0381,0.0684,0.0267,-0.0292,NaN,NaN,-0.0803,True
124,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,5.0,6.0,frozen,5.0,5e-7,0.05,0.0103,0.0044,0.0578,0.0470,-0.0292,-60.5,160.5,0.0120,True
122,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,5.0,6.0,frozen,0.0,5e-7,0.05,-0.0060,-0.0048,0.0547,0.0035,0.0583,92.5,7.5,-0.0147,True



===== Gemma-2 — top 5 by d_cavg (format-controlled) =====


,run,reward,power_k,ll_min,rm_mode,fp,lr,kl,d_hm,d_avg,d_cavg,d_cond_acc,d_fmt_pass,fmt_pct,reason_pct,d_gsm8k,clean
95,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,5.0,5e-7,0.05,0.0751,0.0968,0.1480,0.0906,0.1292,52.7,47.3,0.0563,True
88,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr1e-6_...,log_prob,2.0,8.0,actor,5.0,1e-6,0.05,-0.0026,0.0317,0.1395,0.0869,0.0792,46.0,54.0,0.0230,True
89,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,6.0,frozen,0.0,5e-7,0.05,-0.0398,0.0081,0.0982,0.1044,-0.0208,-29.5,129.5,0.0030,True
113,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,7.0,4.0,actor,5.0,5e-7,0.05,0.0795,0.0756,0.0894,0.0573,0.1583,68.2,31.8,0.0553,True
94,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,5.0,5e-7,0.05,-0.0032,0.0536,0.0719,0.0439,0.0292,36.2,63.8,0.0573,True


### 8c. Archetype aggregation (robust — averages out per-run val-subset noise)
`mean_dcavg` = format-controlled reasoning gain; `mean_dhm` = raw HM gain; `mean_dfmtp` = format-pass gain.


In [11]:
arch2 = (df[df.clean].groupby(['model','reward','rm_mode','fp'])
           .agg(n=('d_cavg','size'), mean_dcavg=('d_cavg','mean'),
                mean_dhm=('d_hm','mean'), mean_dfmtp=('d_fmt_pass','mean')).reset_index())
display(arch2[arch2.n>=3].sort_values('mean_dcavg', ascending=False).round(3))


,model,reward,rm_mode,fp,n,mean_dcavg,mean_dhm,mean_dfmtp
4,Gemma-2,power,actor,5.0,3,0.048,0.041,0.108
6,Gemma-2,power,frozen,5.0,6,0.042,-0.010,0.035
20,Qwen3,power,actor,5.0,6,0.033,-0.001,0.000
7,Qwen2.5,log_prob,actor,0.0,4,0.024,-0.031,-0.052
5,Gemma-2,power,frozen,0.0,5,0.017,0.020,0.036
22,Qwen3,power,frozen,5.0,6,0.009,-0.005,-0.013
12,Qwen2.5,power,actor,5.0,11,0.003,0.040,0.007
9,Qwen2.5,log_prob,frozen,0.0,8,-0.003,-0.107,-0.153
10,Qwen2.5,log_prob,frozen,5.0,8,-0.005,-0.058,-0.107
14,Qwen2.5,power,frozen,5.0,15,-0.005,-0.001,-0.027


## 9. Verdict (eval-grounded + format-controlled)

**Ranking on raw ToM HM (format-confounded):** clean per-family winners share **frozen RM · kl0.05 · fp0** —
Qwen2.5 power-k7-llmin−6-lr5e-7 (dHM +0.055), Qwen3 log_prob-k2-llmin−8-lr1e-6 (+0.063),
Gemma-2 power-k5-llmin−4-lr5e-7 (+0.133). actor-RM blew up KL in 12/72 vs 0/80 frozen; PS129
(actor-k7) had the biggest raw dHM (+0.344) but reward-hacks (kl 9.85, rollout 1.5 tok).

**But most of the raw HM gain is FORMATTING, not reasoning:**
* Gemma-2 winner: format-pass 0.60→0.69 while conditional accuracy is **flat** (0.59→0.57);
  correct-rate split ≈ **145% format / −45% reasoning**; and **95% of its +0.133 HM** comes from
  rescuing 5 floor benchmarks off ~0 (HM over-weights the minimum), which were failing on format.
* PS129: ≈82% format-driven, conditional accuracy flat.
* The Qwen2.5/Qwen3 raw winners show ~0/negative conditional-accuracy gain (small effects, noisy).

**Re-ranking on the format-controlled metric (`d_cavg`, arithmetic-mean accuracy | correct format):**
* No recipe shows a large, robust ToM-reasoning gain; the best signals are small (d_cavg≈0.04–0.15,
  ~1–2 SE) and, in the archetype aggregation, favor **format-penalty (`fp5`) Gemma** configs.
* Most internally-consistent genuine winner: **Gemma-2 power-k5-llmin−4-frozen-`fp5`-ec0.001** —
  positive on all: d_cavg +0.148, d_cond +0.091, d_hm +0.075, gsm8k +0.056.

**Takeaway:** report **conditional accuracy alongside HM** for weak/tag-free bases (Gemma), and
prefer a training **format penalty (`fp5`)** so HM reflects reasoning, not format-compliance.
